# Lesson 2: Interacting with a CSV Data

## Setup LLM (Ollama / OpenAI / Groq / Azure)

**Default: Ollama (gratis, lokal).** Jalankan `python setup_data.py` dulu jika data belum ada.

In [1]:
from llm_setup import get_chat_model, check_llm_config, print_provider_info

issues = check_llm_config()
if issues:
    raise EnvironmentError("\n".join(issues))

print_provider_info()
model = get_chat_model()
print("LLM siap.")

Provider : ollama
Model    : llama3.1:8b
Ollama   : http://localhost:11434
LLM siap.


## Load the dataset

In [2]:
import pandas as pd

df = pd.read_csv("./data/all-states-history.csv").fillna(value=0)
print(f"Baris: {len(df)}, Kolom: {len(df.columns)}")
df.head()

Baris: 20780, Kolom: 41


,date,state,death,deathConfirmed,deathIncrease,deathProbable,hospitalized,hospitalizedCumulative,hospitalizedCurrently,hospitalizedIncrease,...,totalTestResults,totalTestResultsIncrease,totalTestsAntibody,totalTestsAntigen,totalTestsPeopleAntibody,totalTestsPeopleAntigen,totalTestsPeopleViral,totalTestsPeopleViralIncrease,totalTestsViral,totalTestsViralIncrease
0,2021-03-07,AK,305.0,0.0,0,0.0,1293.0,1293.0,33.0,0,...,1731628.0,0,0.0,0.0,0.0,0.0,0.0,0,1731628.0,0
1,2021-03-07,AL,10148.0,7963.0,-1,2185.0,45976.0,45976.0,494.0,0,...,2323788.0,2347,0.0,0.0,119757.0,0.0,2323788.0,2347,0.0,0
2,2021-03-07,AR,5319.0,4308.0,22,1011.0,14926.0,14926.0,335.0,11,...,2736442.0,3380,0.0,0.0,0.0,481311.0,0.0,0,2736442.0,3380
3,2021-03-07,AS,0.0,0.0,0,0.0,0.0,0.0,0.0,0,...,2140.0,0,0.0,0.0,0.0,0.0,0.0,0,2140.0,0
4,2021-03-07,AZ,16328.0,14403.0,5,1925.0,57907.0,57907.0,963.0,44,...,7908105.0,45110,580569.0,0.0,444089.0,0.0,3842945.0,14856,7908105.0,45110


## Prepare the Langchain dataframe agent

In [3]:
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent
from llm_setup import get_agent_type, get_agent_executor_kwargs

agent = create_pandas_dataframe_agent(
    llm=model,
    df=df,
    verbose=True,
    allow_dangerous_code=True,
    agent_type=get_agent_type(),
    agent_executor_kwargs=get_agent_executor_kwargs(),
)

agent.invoke("how many rows are there?")

C:\Users\USER\AppData\Local\Temp\ipykernel_5960\2678910800.py:1: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent




> Entering new AgentExecutor chain...
Thought: I need to find the number of rows in the dataframe `df`.
Action: python_repl_ast
Action Input: `print(df.shape)`(20780, 41)
Thought: The output shows that the shape of the dataframe is (20780, 41), which means there are 20780 rows and 41 columns.

Final Answer: There are 20780 rows in the dataframe `df`.

> Finished chain.


{'input': 'how many rows are there?',
 'output': 'There are 20780 rows in the dataframe `df`.'}

## Design your prompt and ask your question

In [4]:
from prompts import CSV_PROMPT_PREFIX, CSV_PROMPT_SUFFIX

QUESTION = (
    "How many patients were hospitalized during July 2020 "
    "in Texas, and nationwide as the total of all states? "
    "Use the hospitalizedIncrease column"
)

agent.invoke(CSV_PROMPT_PREFIX + QUESTION + CSV_PROMPT_SUFFIX)



> Entering new AgentExecutor chain...
Thought: I need to set the pandas display options to show all columns.
Action: python_repl_ast
Action Input: `pd.set_option('display.max_columns', None)`NameError: name 'pd' is not definedIt seems like we haven't imported the pandas library yet. Let's fix that.

Thought: We need to import the pandas library.
Action: python_repl_ast
Action Input: `import pandas as pd`Thought: Now that we have imported the pandas library, let's set the display options to show all columns.
Action: python_repl_ast
Action Input: `pd.set_option('display.max_columns', None)`Thought: We need to get the column names of the dataframe.
Action: python_repl_ast
Action Input: `print(df.columns.tolist())`['date', 'state', 'death', 'deathConfirmed', 'deathIncrease', 'deathProbable', 'hospitalized', 'hospitalizedCumulative', 'hospitalizedCurrently', 'hospitalizedIncrease', 'inIcuCumulative', 'inIcuCurrently', 'negative', 'negativeIncrease', 'negativeTestsAntibody', 'negativeTests

{'input': '\nFirst set the pandas display options to show all the columns,\nget the column names, then answer the question.\nHow many patients were hospitalized during July 2020 in Texas, and nationwide as the total of all states? Use the hospitalizedIncrease column\n- **ALWAYS** before giving the Final Answer, try another method.\nThen reflect on the answers of the two methods you did and ask yourself\nif it answers correctly the original question.\nIf you are not sure, try another method.\n- If the methods tried do not give the same result,reflect and\ntry again until you have two methods that have the same result.\n- If you still cannot arrive to a consistent result, say that\nyou are not sure of the answer.\n- If you are sure of the correct answer, create a beautiful\nand thorough response using Markdown.\n- **DO NOT MAKE UP AN ANSWER OR USE PRIOR KNOWLEDGE,\nONLY USE THE RESULTS OF THE CALCULATIONS YOU HAVE DONE**.\n- **ALWAYS**, as part of your "Final Answer", explain how you got